In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Data
users = spark.createDataFrame([
    ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg")
], ["user_id", "city"])

orders = spark.createDataFrame([
    ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0)
], ["order_id", "user_id", "product_id", "qty", "price"])

products = spark.createDataFrame([
    ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA")
], ["product_id", "product_name"])

# Revenue
orders_with_revenue = orders.withColumn("revenue", F.col("qty") * F.col("price"))

# Join
df_joined = orders_with_revenue \
    .join(users, "user_id") \
    .join(products, "product_id")

# GroupBy
city_product_stats = df_joined.groupBy("city", "product_id", "product_name") \
    .agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum")
    )

# Top-2
window_spec = Window.partitionBy("city").orderBy(F.desc("revenue_sum"))
city_top_products = city_product_stats \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 2) \
    .drop("rank")

# HDFS
output_path = "/tmp/sandbox_zeppelin/mart_city_top_products/"
city_top_products.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_path)

# Result
df_check = spark.read.format("parquet").load(output_path)
df_check.orderBy("city", F.desc("revenue_sum")).show(truncate=False)